Purpose: EXTRACT.

Read raw retail data from CSV into a pandas DataFrame for downstream processing.

Method:

Uses pd.read_csv() with:
|Explicit encoding.
|Automatic datetime parsing for InvoiceDate
|Type hints for critical columns (e.g., CustomerID as string)

-Logs extraction status and row count

-Returns a DataFrame for transformation

In [2]:
import pandas as pd
import logging
from datetime import datetime
import sqlite3


def extract_data(file_path: str) -> pd.DataFrame:
    """
    Extract Online Retail data from CSV.
    Handles date parsing and initial data validation.
    """
    try:
        df = pd.read_csv(
            file_path,
            encoding='ISO-8859-1',
            parse_dates=['InvoiceDate'],
            dtype={
                'InvoiceNo': str,
                'StockCode': str,
                'Description': str,
                'CustomerID': str,
                'Country': str
            }
        )
        logging.info(f"Successfully extracted {len(df)} rows from {file_path}")
        return df
    except Exception as e:
        logging.error(f"Extraction failed: {str(e)}")
        raise

Purpose: TRANSFORM.

Clean, validate, and structure data into fact/dimension tables for the star schema.

Method:

Data Cleaning:

-Filters invalid records (Quantity > 0, UnitPrice > 0)

-Drops rows with missing CustomerID

Feature Engineering:

-Adds TotalSales (Quantity × UnitPrice)

-Creates InvoiceYearMonth for time analysis

Time Filtering:

-Keeps only the last year of data (relative to 2025-08-12)

Dimension Creation:

-CustomerDim: Aggregates by CustomerID (country, total spend, transaction count)

-TimeDim: Extracts date parts (day/month/year) from InvoiceDate

Logs row counts for each output table

In [3]:

def transform_data(df: pd.DataFrame, current_date: str = '2025-08-12') -> tuple:
    """
    Transform raw data into fact and dimension tables.
    Returns (fact_df, customer_dim, time_dim)
    """
    current_date = pd.to_datetime(current_date)
    
    # Data cleaning
    df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
    df = df.dropna(subset=['CustomerID'])
    
    # Calculate derived columns
    df['TotalSales'] = df['Quantity'] * df['UnitPrice']
    df['InvoiceYearMonth'] = df['InvoiceDate'].dt.to_period('M')
    
    # Filter for last year
    one_year_ago = current_date - pd.DateOffset(years=1)
    df = df[df['InvoiceDate'] >= one_year_ago]
    
    # Create dimension tables
    customer_dim = df.groupby('CustomerID').agg({
        'Country': 'first',
        'TotalSales': 'sum',
        'InvoiceNo': 'nunique'
    }).reset_index().rename(columns={
        'InvoiceNo': 'TotalTransactions'
    })
    
    time_dim = df[['InvoiceDate']].drop_duplicates()
    time_dim['DateKey'] = time_dim['InvoiceDate'].dt.strftime('%Y%m%d')
    time_dim['Day'] = time_dim['InvoiceDate'].dt.day
    time_dim['Month'] = time_dim['InvoiceDate'].dt.month
    time_dim['Year'] = time_dim['InvoiceDate'].dt.year
    time_dim['Quarter'] = time_dim['InvoiceDate'].dt.quarter
    time_dim['DayOfWeek'] = time_dim['InvoiceDate'].dt.dayofweek
    
    # Prepare fact table
    fact_df = df[[
        'InvoiceNo', 'StockCode', 'Description',
        'Quantity', 'UnitPrice', 'TotalSales',
        'CustomerID', 'InvoiceDate'
    ]]
    
    logging.info(f"Transformed data: {len(fact_df)} fact records, "
                 f"{len(customer_dim)} customers, {len(time_dim)} dates")
    return fact_df, customer_dim, time_dim

Purpose: LOAD.

-Persist transformed data to SQLite with a star schema.

Method:

Database Setup:

-Executes ddl.sql to create tables with proper constraints

Data Loading:

-Uses pd.to_sql() to bulk-insert:
 .Fact table (SalesFact)
 .Dimensions (CustomerDim, TimeDim)

Optimization:

-Adds indexes on join keys (CustomerID, InvoiceDate)

Transaction Safety:

-Wrapped in try-catch with connection cleanup

In [4]:

def load_data(fact_df: pd.DataFrame, customer_dim: pd.DataFrame, 
              time_dim: pd.DataFrame, db_path: str) -> None:
    """
    Load data into SQLite database with star schema
    """
    try:
        conn = sqlite3.connect(db_path)
        
        # Create tables
        with open('sql/ddl.sql', 'r') as f:
            conn.executescript(f.read())
        
        # Load data
        fact_df.to_sql('SalesFact', conn, if_exists='append', index=False)
        customer_dim.to_sql('CustomerDim', conn, if_exists='append', index=False)
        time_dim.to_sql('TimeDim', conn, if_exists='append', index=False)
        
        logging.info(f"Successfully loaded data to {db_path}")
    except Exception as e:
        logging.error(f"Loading failed: {str(e)}")
        raise
    finally:
        if conn:
            conn.close()

Purpose:

Orchestrate the end-to-end ETL pipeline with logging.

Method:

Sequence:

extract_data() → transform_data() → load_data()
Logging:

Tracks progress (row counts, errors) at each stage

Uses timestamped messages for debugging

Entry Point:

CLI-ready script with hardcoded paths (configurable via arguments)

In [9]:

def run_etl(input_path: str, output_path: str) -> None:
    """Complete ETL pipeline with logging"""
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s'
    )
    
    try:
        logging.info("Starting ETL process")
        
        # Extract
        raw_data = extract_data(input_path)
        logging.info(f"Extracted {len(raw_data)} raw records")
        
        # Transform
        fact, customers, time = transform_data(raw_data)
        logging.info(f"Transformed data: {len(fact)} facts, "
                    f"{len(customers)} customers, {len(time)} dates")
        
        # Load
        load_data(fact, customers, time, output_path)
        logging.info("ETL completed successfully")
        
    except Exception as e:
        logging.error(f"ETL failed: {str(e)}")
        raise

if __name__ == "__main__":
    run_etl(
        input_path='data/OnlineRetail.csv',
        output_path='data/retail_dw.db'
    )

2025-08-13 11:11:12,764 - INFO - Starting ETL process
2025-08-13 11:11:12,777 - ERROR - Extraction failed: [Errno 2] No such file or directory: 'data/OnlineRetail.csv'
2025-08-13 11:11:12,778 - ERROR - ETL failed: [Errno 2] No such file or directory: 'data/OnlineRetail.csv'


FileNotFoundError: [Errno 2] No such file or directory: 'data/OnlineRetail.csv'